# Analyse croisée des flux logistiques
Objectif : repérer des écarts vérifiables entre ajustements, réceptions, commandes et expéditions, puis proposer des vérifications au groupe. Un écart n'est pas une preuve de fraude ou de causalité.

Les CSV nettoyés restent intacts. Les rapprochements ci-dessous sont des vues d'analyse. Les absences de référence sont distinguées des références introuvables ; les valeurs inconnues ne deviennent pas artificiellement des incohérences.

## 1. Sources et granularité
Une ligne représente un ajustement, une réception, une ligne de commande ou une étape d'expédition selon la table. Les clés sont vérifiées avant chaque rapprochement. Aucune référence dupliquée n'est supprimée arbitrairement.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Fonctionne depuis la racine du dépôt ou le dossier du notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data/logistics').is_dir()), None)
assert ROOT is not None, "Ouvrir le notebook depuis le dépôt."
DATA = ROOT / 'data/logistics'
files = {
    'adjustments': 'inventory_adjustments_cleaned.csv',
    'receipts': 'goods_receipts_cleaned.csv',
    'orders': 'purchase_order_lines_clean.csv',
    'shipments': 'shipments_clean.csv',
    'facilities': 'erp_facilities_cleaned.csv',
    'partners': 'erp_business_partners_cleaned.csv',
}
tables = {name: pd.read_csv((DATA / filename if (DATA / filename).exists() else DATA / 'cleaned' / filename)) for name, filename in files.items()}
a, r, o, s, f, p = [tables[name].copy() for name in files]
display(pd.DataFrame([
    {'table': name, 'fichier': files[name], 'lignes': len(df), 'colonnes': len(df.columns)}
    for name, df in tables.items()
]))


         table                            fichier  lignes  colonnes
0  adjustments  inventory_adjustments_cleaned.csv     140        11
1     receipts         goods_receipts_cleaned.csv     610        12
2       orders     purchase_order_lines_clean.csv     650        19
3    shipments                shipments_clean.csv     450        19
4   facilities         erp_facilities_cleaned.csv      29        11
5     partners  erp_business_partners_cleaned.csv      80        11

In [2]:
keys = {
    'adjustments': ['adjustment_number'], 'receipts': ['receipt_number'],
    'orders': ['purchase_order_number', 'purchase_order_line'],
    'shipments': ['shipment_number', 'shipment_leg'],
    'facilities': ['facility_code'], 'partners': ['partner_code'],
}
for name, columns in keys.items():
    assert tables[name][columns].notna().all().all(), (name, 'clé vide')
    assert not tables[name].duplicated(columns).any(), (name, 'clé dupliquée')
for df, columns in [
    (a, ['adjustment_date']), (r, ['receipt_date']),
    (o, ['order_date', 'requested_delivery_date']),
    (s, ['dispatch_time', 'arrival_time']),
]:
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors='raise')
for df, column in [(a, 'adjustment_quantity'), (r, 'received_quantity'), (o, 'ordered_quantity')]:
    df[column] = pd.to_numeric(df[column], errors='raise')
print("Clés uniques au niveau prévu ; dates et quantités convertibles.")
print("Expéditions à plusieurs étapes :", int(s['shipment_number'].duplicated().sum()))


Clés uniques au niveau prévu ; dates et quantités convertibles.
Expéditions à plusieurs étapes : 0


## 2. Couverture des références
Les jointures sont gauches : chaque événement initial est conservé. Une référence vide signifie « non renseignée », pas nécessairement une erreur. Une référence renseignée mais introuvable nécessite une vérification.

Le référentiel nettoyé des établissements est confronté au brut uniquement pour repérer les codes disparus pendant le nettoyage ; aucune correction automatique n'est faite.

In [3]:
coverage = []
for label, source, target in [
    ('Ajustement → établissement', a['facility_code'], f['facility_code']),
    ('Ajustement → réception', a['related_receipt_number'], r['receipt_number']),
    ('Ajustement → expédition', a['related_shipment_number'], s['shipment_number']),
    ('Réception → établissement', r['receiving_facility_code'], f['facility_code']),
    ('Réception → fournisseur', r['supplier_partner_code'], p['partner_code']),
]:
    present = source.notna()
    coverage.append({'relation': label, 'lignes': len(source),
                     'non_renseignées': int((~present).sum()),
                     'retrouvées': int((present & source.isin(target)).sum()),
                     'introuvables': int((present & ~source.isin(target)).sum())})
display(pd.DataFrame(coverage))
raw_f = pd.read_csv(DATA / 'erp_facilities.csv')
missing_f = raw_f.loc[~raw_f['facility_code'].isin(f['facility_code']),
                      ['facility_code', 'facility_name', 'facility_category']]
display(missing_f)


                     relation  lignes  non_renseignées  retrouvées  introuvables
0  Ajustement → établissement     140                0         136             4
1      Ajustement → réception     140              117          23             0
2     Ajustement → expédition     140              103          37             0
3   Réception → établissement     610                0         590            20
4     Réception → fournisseur     610                0         610             0

  facility_code                 facility_name facility_category
8      FCL-1029  Vale Strategy Valenne Office            office

## 3. Commandes et réceptions
Jointure sur **numéro de commande + numéro de ligne**. Plusieurs réceptions peuvent correspondre à une ligne de commande : on contrôle aussi leur total avant de parler de réception partielle ou excédentaire.

Un délai après la date demandée est un constat, pas une faute établie. Les quantités ne sont comparées que lorsque les unités correspondent. Les références sans réception sont conservées dans le bilan des commandes.

In [4]:
po_keys = ['purchase_order_number', 'purchase_order_line']
receipt_orders = r.merge(o, on=po_keys, how='left', suffixes=('_receipt', '_order'),
                         validate='many_to_one', indicator='order_match')
assert len(receipt_orders) == len(r)

def different_known(left, right):
    result = left.ne(right).astype('boolean')
    return result.mask(left.isna() | right.isna(), pd.NA)

receipt_orders['supplier_diff'] = different_known(
    receipt_orders['supplier_partner_code_receipt'], receipt_orders['supplier_partner_code_order'])
receipt_orders['facility_diff'] = different_known(
    receipt_orders['receiving_facility_code'], receipt_orders['ordering_facility_code'])
receipt_orders['unit_diff'] = different_known(receipt_orders['received_unit'], receipt_orders['ordered_unit'])
receipt_orders['days_after_order'] = (
    receipt_orders['receipt_date'] - receipt_orders['order_date']).dt.days
receipt_orders['days_after_requested'] = (
    receipt_orders['receipt_date'] - receipt_orders['requested_delivery_date']).dt.days
display(pd.DataFrame({
    'contrôle': ['Commande introuvable', 'Fournisseur différent', 'Établissement différent',
                 'Unité différente', 'Réception avant commande', 'Réception après date demandée'],
    'nombre': [receipt_orders['order_match'].eq('left_only').sum(),
               receipt_orders['supplier_diff'].sum(), receipt_orders['facility_diff'].sum(),
               receipt_orders['unit_diff'].sum(), receipt_orders['days_after_order'].lt(0).sum(),
               receipt_orders['days_after_requested'].gt(0).sum()]
}))
display(receipt_orders[['days_after_order', 'days_after_requested']].describe())


                        contrôle  nombre
0           Commande introuvable       0
1          Fournisseur différent       0
2        Établissement différent       0
3               Unité différente       0
4       Réception avant commande       0
5  Réception après date demandée     386

       days_after_order  days_after_requested
count        610.000000            610.000000
mean          25.952459              1.973770
std           10.549408              3.287929
min            4.000000            -22.000000
25%           17.000000             -1.000000
50%           26.000000              2.000000
75%           35.000000              5.000000
max           48.000000              7.000000

In [5]:
received_totals = receipt_orders.groupby(po_keys).agg(
    n_receipts=('receipt_number', 'size'),
    total_received=('received_quantity', lambda x: x.sum(min_count=1)),
    quantities_known=('received_quantity', lambda x: x.notna().all()),
    units_comparable=('unit_diff', lambda x: x.notna().all() and not x.any()),
).reset_index()
order_balance = o.merge(received_totals, on=po_keys, how='left', validate='one_to_one')
order_balance['n_receipts'] = order_balance['n_receipts'].fillna(0).astype(int)
comparable = (order_balance['n_receipts'].gt(0)
              & order_balance['units_comparable'].eq(True)
              & order_balance['quantities_known'].eq(True)
              & order_balance['ordered_quantity'].notna())
order_balance['quantity_gap'] = (
    order_balance['total_received'] - order_balance['ordered_quantity']).where(comparable)
tol = 1e-6
print('Lignes de commande sans réception dans les données :', order_balance['n_receipts'].eq(0).sum())
print('Lignes avec comparaison quantitative possible :', comparable.sum())
print('Total reçu inférieur à la commande :', order_balance['quantity_gap'].lt(-tol).sum())
print('Total reçu supérieur à la commande :', order_balance['quantity_gap'].gt(tol).sum())
display(order_balance.loc[order_balance['quantity_gap'].lt(-tol),
    po_keys + ['item_category', 'ordered_unit', 'ordered_quantity', 'total_received',
               'quantity_gap', 'order_status']].head(12))


Lignes de commande sans réception dans les données : 40
Lignes avec comparaison quantitative possible : 610
Total reçu inférieur à la commande : 65
Total reçu supérieur à la commande : 0


    purchase_order_number  purchase_order_line           item_category ordered_unit  ordered_quantity  total_received  quantity_gap order_status
1           PO-2015-00001                    1       transport_service      service                 1            0.90         -0.10    completed
28          PO-2016-00002                    1     medical_consumables         case                61           56.41         -4.59    completed
33          PO-2016-00007                    1   refrigeration_service      service                 1            0.89         -0.11    completed
41          PO-2016-00015                    1     medical_consumables         case                 3            2.44         -0.56    completed
54          PO-2016-00028                    1       transport_service      service                 1            0.88         -0.12    completed
57          PO-2017-00003                    1       biological_sample   sealed_lot                 2            1.66         -0.3

## 4. Ajustements et réceptions : preuves ligne par ligne
Une quantité d'ajustement négative est normale pour une diminution de stock. On ne la corrige pas en fonction du motif.

La catégorie de la réception est obtenue via sa ligne de commande. Un écart d'établissement ou d'unité indique une différence observable ; la nature du lien peut éventuellement l'expliquer. Une réception future est calculée à partir de sa date effective, pas de l'année de son identifiant.

In [6]:
receipt_ref = receipt_orders[[
    'receipt_number', 'receipt_date', 'receiving_facility_code', 'received_unit',
    'received_quantity', 'item_category', 'purchase_order_number', 'purchase_order_line',
    'supplier_partner_code_receipt'
]].rename(columns={'item_category': 'receipt_item_category',
                   'supplier_partner_code_receipt': 'receipt_supplier_code'})
ar = a.merge(receipt_ref, left_on='related_receipt_number', right_on='receipt_number',
             how='left', validate='many_to_one', indicator='receipt_match')
assert len(ar) == len(a)
ar['receipt_days_after_adjustment'] = (ar['receipt_date'] - ar['adjustment_date']).dt.days
ar['receipt_future'] = ar['receipt_days_after_adjustment'].gt(0).astype('boolean').mask(
    ar['receipt_days_after_adjustment'].isna())
ar['receipt_facility_diff'] = different_known(ar['facility_code'], ar['receiving_facility_code'])
ar['receipt_unit_diff'] = different_known(ar['unit'], ar['received_unit'])
ar['receipt_category_diff'] = different_known(ar['item_category'], ar['receipt_item_category'])
receipt_flags = ['receipt_future', 'receipt_facility_diff', 'receipt_unit_diff', 'receipt_category_diff']

def summarize_flags(df, columns):
    return pd.DataFrame([{'indicateur': c, 'écarts': int(df[c].sum()),
                          'comparables': int(df[c].notna().sum()),
                          'non_évaluables': int(df[c].isna().sum())} for c in columns])
display(summarize_flags(ar, receipt_flags))
receipt_evidence = ar.loc[ar[receipt_flags].fillna(False).any(axis=1), [
    'adjustment_number', 'adjustment_date', 'facility_code', 'item_category', 'unit',
    'adjustment_reason', 'related_receipt_number', 'receipt_date', 'receiving_facility_code',
    'receipt_item_category', 'received_unit', 'receipt_days_after_adjustment', *receipt_flags
]].sort_values('receipt_days_after_adjustment', ascending=False)
display(receipt_evidence.head(15))


              indicateur  écarts  comparables  non_évaluables
0         receipt_future      11           23             117
1  receipt_facility_diff      21           23             117
2      receipt_unit_diff      21           23             117
3  receipt_category_diff      22           23             117

   adjustment_number adjustment_date facility_code         item_category        unit            adjustment_reason related_receipt_number receipt_date receiving_facility_code   receipt_item_category  \
5     ADJ-2016-00006      2016-06-12      FCL-1004   medical_consumables        case  damaged packaging write-off         RCV-2024-00527   2024-07-04                FCL-1016    agricultural_product   
2     ADJ-2015-00003      2015-10-30      FCL-1024  monitoring_equipment         kit         late receipt posting         RCV-2022-00361   2022-08-25                FCL-1011                   other   
15    ADJ-2018-00016      2018-02-22      FCL-1029     biological_sample  sealed_lot       cycle count correction         RCV-2024-00522   2024-07-06                FCL-1011       medical_equipment   
17    ADJ-2018-00018      2018-02-26      FCL-1029     biological_sample  sealed_lot       cycle count correction         RCV-2023-00446   2023-09-15                FCL-1030    protective_equipmen

## 5. Ajustements et expéditions
Une expédition peut avoir plusieurs étapes. On conserve toutes les étapes dans une table de détail, puis on ramène les indicateurs à **un résultat par ajustement**.

Un établissement est dit absent du trajet s'il n'apparaît à aucune étape connue. Une catégorie ou unité est dite incompatible si aucune étape ne correspond. Le départ est futur si le premier départ est postérieur au jour de l'ajustement : l'heure de l'ajustement étant inconnue, on ne conclut pas pour un départ le même jour.

In [7]:
shipment_columns = ['shipment_number', 'shipment_leg', 'dispatch_time', 'arrival_time',
                    'origin_facility_code', 'destination_facility_code',
                    'item_category', 'declared_unit', 'declared_quantity', 'shipment_status']
shipment_detail = a.loc[a['related_shipment_number'].notna()].merge(
    s[shipment_columns], left_on='related_shipment_number', right_on='shipment_number',
    how='left', suffixes=('_adjustment', '_shipment'), validate='many_to_many',
    indicator='shipment_match')
rows = []
for adj_id, group in shipment_detail.groupby('adjustment_number', sort=False):
    found = group['shipment_match'].eq('both').all()
    def no_match(target, columns):
        values = group[columns]
        if not found or pd.isna(target) or values.isna().any().any():
            return pd.NA
        return not values.eq(target).any().any()
    first_dispatch = group['dispatch_time'].min()
    future = pd.NA
    if found and group['dispatch_time'].notna().all() and pd.notna(group['adjustment_date'].iloc[0]):
        future = first_dispatch.normalize() > group['adjustment_date'].iloc[0].normalize()
    rows.append({
        'adjustment_number': adj_id, 'shipment_found': found,
        'n_shipment_legs': int(group['shipment_number'].notna().sum()),
        'first_dispatch': first_dispatch,
        'shipment_future': future,
        'shipment_facility_diff': no_match(group['facility_code'].iloc[0],
                                          ['origin_facility_code', 'destination_facility_code']),
        'shipment_category_diff': no_match(group['item_category_adjustment'].iloc[0],
                                          ['item_category_shipment']),
        'shipment_unit_diff': no_match(group['unit'].iloc[0], ['declared_unit']),
    })
shipment_summary = pd.DataFrame(rows)
shipment_flags = ['shipment_future', 'shipment_facility_diff', 'shipment_category_diff', 'shipment_unit_diff']
for column in shipment_flags:
    shipment_summary[column] = shipment_summary[column].astype('boolean')
as_ = a.merge(shipment_summary, on='adjustment_number', how='left', validate='one_to_one')
assert len(as_) == len(a)
display(summarize_flags(as_, shipment_flags))
display(shipment_detail.loc[
    shipment_detail['adjustment_number'].isin(
        shipment_summary.loc[shipment_summary[shipment_flags].fillna(False).any(axis=1), 'adjustment_number']),
    ['adjustment_number', 'adjustment_date', 'related_shipment_number', 'shipment_leg',
     'dispatch_time', 'facility_code', 'origin_facility_code', 'destination_facility_code',
     'item_category_adjustment', 'item_category_shipment', 'unit', 'declared_unit']
].head(15))


               indicateur  écarts  comparables  non_évaluables
0         shipment_future      13           37             103
1  shipment_facility_diff      18           37             103
2  shipment_category_diff      17           37             103
3      shipment_unit_diff      17           37             103

   adjustment_number adjustment_date related_shipment_number  shipment_leg       dispatch_time facility_code origin_facility_code destination_facility_code item_category_adjustment  \
0     ADJ-2018-00014      2018-02-18          SHP-2023-00029             1 2023-04-22 09:00:00      FCL-1027             FCL-1018                  FCL-1011     agricultural_product   
1     ADJ-2018-00016      2018-02-22          SHP-2025-00012             1 2025-05-05 16:00:00      FCL-1029             FCL-1020                  FCL-1015        biological_sample   
2     ADJ-2018-00017      2018-02-23          SHP-2024-00012             1 2024-06-01 05:00:00      FCL-1019             FCL-1021                  FCL-1008     agricultural_product   
3     ADJ-2018-00018      2018-02-26          SHP-2023-00037             1 2023-09-01 05:00:00      FCL-1029             FCL-1020                  FCL-1011        biological_sample   
8     ADJ-2019-00041      2019-09-12          SHP-2021-00017             1 2021-

## 6. Où et quand se concentrent les écarts ?
Chaque ajustement est compté une seule fois, même si plusieurs indicateurs sont positifs. Les nombres sont accompagnés du nombre de lignes avec référence et du nombre réellement évaluable : un établissement plus documenté peut présenter davantage d'écarts.

Les regroupements ne constituent pas un classement de culpabilité. Les petits effectifs et l'incomplétude des sources limitent les comparaisons.

In [8]:
review = ar.merge(shipment_summary, on='adjustment_number', how='left', validate='one_to_one')
review = review.merge(f[['facility_code', 'facility_name', 'region']],
                      on='facility_code', how='left', validate='many_to_one')
review['facility_name'] = review['facility_name'].fillna('[absent du référentiel nettoyé]')
all_flags = receipt_flags + shipment_flags
review['has_reference'] = review[['related_receipt_number', 'related_shipment_number']].notna().any(axis=1)
review['evaluable'] = review[all_flags].notna().any(axis=1)
review['any_gap'] = review[all_flags].fillna(False).any(axis=1)
review['year'] = review['adjustment_date'].dt.year
assert len(review) == len(a) and review['adjustment_number'].is_unique

def concentration(group_keys):
    result = review.groupby(group_keys, dropna=False).agg(
        adjustments=('adjustment_number', 'size'),
        with_reference=('has_reference', 'sum'),
        evaluable=('evaluable', 'sum'),
        with_gap=('any_gap', 'sum')
    )
    result['pct_with_gap_among_evaluable'] = (
        result['with_gap'] / result['evaluable'].replace(0, float('nan')) * 100).round(1)
    return result.sort_values(['with_gap', 'adjustments'], ascending=False)

display(concentration(['facility_code', 'facility_name']))
display(concentration(['year']).sort_index())
display(concentration(['adjustment_reason']))


                                                                adjustments  with_reference  evaluable  with_gap  pct_with_gap_among_evaluable
facility_code facility_name                                                                                                                   
FCL-1023      North Quay Biomedical Store                                 6               4          4         4                         100.0
FCL-1008      Civic Health Receiving Centre                               5               3          3         3                         100.0
FCL-1029      [absent du référentiel nettoyé]                             4               3          3         3                         100.0
FCL-1009      Nordhaven Freight Exchange Warehouse                        3               3          3         3                         100.0
FCL-1026      Amber Kintara Transit Warehouse                             3               3          3         3                         100.0

      adjustments  with_reference  evaluable  with_gap  pct_with_gap_among_evaluable
year                                                                                
2014            1               0          0         0                          <NA>
2015            2               1          1         1                         100.0
2016            5               1          1         1                         100.0
2017            3               0          0         0                          <NA>
2018           18               7          7         4                          57.1
2019           23               9          9         6                          66.7
2020           14               7          7         5                          71.4
2021           15               7          7         5                          71.4
2022           17               7          7         6                          85.7
2023           13               5          5         3           

                             adjustments  with_reference  evaluable  with_gap  pct_with_gap_among_evaluable
adjustment_reason                                                                                          
cycle count correction                23               9          9         9                         100.0
quality-hold release                  21              11         11         8                          72.7
damaged packaging write-off           16               6          6         6                         100.0
unit-of-measure review                12               6          6         6                         100.0
transfer posting correction           24              14         14         4                          28.6
late receipt posting                   9               4          4         4                         100.0
specialist programme issue            11               0          0         0                          <NA>
procedure stock issue       

## 7. Tableau de travail et pistes à vérifier
Les preuves détaillées sont dans `receipt_evidence`, `shipment_detail` et `review`. On peut filtrer ces tables par identifiant sans perdre le contexte de l'événement.

Les retards et quantités reçues inférieures aux commandes peuvent correspondre à des livraisons partielles, des commandes ouvertes ou une extraction incomplète. Les références futures peuvent résulter d'une erreur de lien, d'une association ajoutée après coup ou d'une autre convention documentaire. Aucune de ces explications n'est prouvée ici.

In [9]:
print('Ajustements au total :', len(review))
print('Avec au moins une référence :', int(review['has_reference'].sum()))
print('Avec au moins un écart mesurable :', int(review['any_gap'].sum()))
print('Réceptions futures (date exacte) :', int(review['receipt_future'].sum()))
print('Expéditions futures (premier départ, jour entier) :', int(review['shipment_future'].sum()))
display(review.loc[review['any_gap'], [
    'adjustment_number', 'adjustment_date', 'facility_code', 'facility_name',
    'adjustment_reason', 'adjustment_status', 'related_receipt_number',
    'related_shipment_number', *all_flags
]].sort_values('adjustment_date').head(20))

hypotheses = pd.DataFrame([
    {
        'piste': 'Références vers des événements futurs',
        'observation': f"{int(review['receipt_future'].sum())} liens réception ; {int(review['shipment_future'].sum())} liens expédition",
        'alternative': 'Erreur de référence ou association rétrospective',
        'à vérifier': 'Rechercher les identifiants dans les notes et documents ; préciser le sens de related_*'
    },
    {
        'piste': 'Différences d’établissement, catégorie ou unité',
        'observation': 'Voir les tables de preuves et les indicateurs séparés',
        'alternative': 'Transfert, regroupement ou conversion documentée',
        'à vérifier': 'Lire les motifs et documents liés ; ne pas comparer des quantités de catégories différentes'
    },
    {
        'piste': 'Concentration de certains écarts',
        'observation': 'Tableaux par établissement, année et motif avec dénominateurs',
        'alternative': 'Volume ou couverture des données différents',
        'à vérifier': 'Confronter aux documents administratifs, puis aux autres domaines si une clé fiable existe'
    },
    {
        'piste': 'Couverture du référentiel nettoyé',
        'observation': ', '.join(missing_f['facility_code'].astype(str)),
        'alternative': 'Suppression volontaire ou involontaire pendant le nettoyage',
        'à vérifier': 'Discuter du traitement de ces établissements avec l’auteur du notebook facilities'
    },
])
display(hypotheses)


Ajustements au total : 140
Avec au moins une référence : 55
Avec au moins un écart mesurable : 37
Réceptions futures (date exacte) : 11
Expéditions futures (premier départ, jour entier) : 13


   adjustment_number adjustment_date facility_code                                     facility_name            adjustment_reason adjustment_status related_receipt_number related_shipment_number  \
2     ADJ-2015-00003      2015-10-30      FCL-1024  Meridian Kintara Biological Aggregation Facility         late receipt posting            posted         RCV-2022-00361                     NaN   
5     ADJ-2016-00006      2016-06-12      FCL-1004                  Kestrel Conservation Field Depot  damaged packaging write-off            posted         RCV-2024-00527                     NaN   
13    ADJ-2018-00014      2018-02-18      FCL-1027               Valenne Central Distribution Centre         quality-hold release          approved         RCV-2023-00414          SHP-2023-00029   
15    ADJ-2018-00016      2018-02-22      FCL-1029                   [absent du référentiel nettoyé]       cycle count correction            posted         RCV-2024-00522          SHP-2025-00012   
16    ADJ-

                                             piste                                        observation                                        alternative  \
0            Références vers des événements futurs           11 liens réception ; 13 liens expédition   Erreur de référence ou association rétrospective   
1  Différences d’établissement, catégorie ou unité  Voir les tables de preuves et les indicateurs ...   Transfert, regroupement ou conversion documentée   
2                 Concentration de certains écarts  Tableaux par établissement, année et motif ave...        Volume ou couverture des données différents   
3                Couverture du référentiel nettoyé                                           FCL-1029  Suppression volontaire ou involontaire pendant...   

                                          à vérifier  
0  Rechercher les identifiants dans les notes et ...  
1  Lire les motifs et documents liés ; ne pas com...  
2  Confronter aux documents administratifs, puis ...  

## Limites et suite proposée au groupe
- Les contrôles portent sur les fichiers nettoyés disponibles ; leurs propres choix de nettoyage peuvent influencer les résultats.
- Les références absentes ne permettent aucune comparaison. Une absence d'écart observé n'est pas une preuve de cohérence complète.
- Aucun seuil statistique ni score de suspicion n'est utilisé. Les indicateurs ne sont ni indépendants ni des preuves d'intention.
- Les chronologies d'expédition portent sur le départ, pas sur une obligation de livraison. Les relations « related » restent à interpréter.
- Les quantités d'unités différentes ne sont jamais additionnées dans un indicateur global.
- Prochaine étape : sélectionner quelques cas documentés, rechercher leurs identifiants dans les textes et confronter plusieurs explications. Les stocks mensuels peuvent ensuite faire l'objet d'une analyse séparée, après clarification des conventions comptables.
- Assistance IA : ce notebook a été préparé avec une assistance au code et à la rédaction. Les hypothèses et conclusions doivent être relues et discutées par le groupe.
